# Siege Utilities: Hydra + Pydantic Configuration System Demo

This notebook demonstrates the new advanced configuration system with Hydra composition and Pydantic validation.

## Features Demonstrated
- **Type-safe configuration management** with full IDE support
- **Client-specific customization** with hierarchical overrides
- **Comprehensive validation** with detailed error messages
- **Seamless migration** from legacy configuration systems

## What this shows
How `siege_utilities.config` loads layered Hydra + Pydantic settings from YAML, env vars, and overrides.

## Why it matters
Every other module in this library depends on config — master this first.

## Prereqs
- `pip install 'siege-utilities[config]'`

## Next
- `02_Create_User_Client_Profiles.ipynb` — the profile system that sits on top of config.


## 1. Basic Setup and Imports

In [ ]:
from pathlib import Path
from siege_utilities.config import HydraConfigManager
from siege_utilities.config import UserProfile, ClientProfile, BrandingConfig, ReportPreferences, ContactInfo
import siege_utilities as su

# Initialize logging (console-only, no file needed for notebooks)
su.configure_shared_logging(level="INFO")

# --- Output configuration ---
OUTPUT_DIR = Path("output") / "notebook_01"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

su.log_info(f"Output directory: {OUTPUT_DIR}  (exists={OUTPUT_DIR.exists()})")
su.log_info("Configuration system demo initialized")

## 2. Loading Configuration with HydraConfigManager

The HydraConfigManager provides a context manager for loading hierarchical configurations.

In [ ]:
# Load configurations using HydraConfigManager
with HydraConfigManager() as manager:
    # Load user profile (uses defaults if no config file exists)
    user_profile = manager.load_user_profile()
    su.log_info(f"User Profile Loaded: {user_profile.username}")
    su.log_debug(f"  Email: {user_profile.email}")
    su.log_debug(f"  Default format: {user_profile.default_output_format}")
    
    # Load default branding configuration
    branding = manager.load_branding_config()
    su.log_info(f"Default Branding Loaded")
    su.log_debug(f"  Primary color: {branding.primary_color}")
    su.log_debug(f"  Primary font: {branding.primary_font}")

## 3. Creating Profiles Programmatically

You can create profiles with full type checking and validation.

In [ ]:
# Create a user profile with custom settings
custom_user = UserProfile(
    username="dheerajchand",
    email="dheeraj@siegeanalytics.com",
    full_name="Dheeraj Chand",
    organization="Siege Analytics",
    default_output_format="pdf",
    default_dpi=300,
    preferred_map_style="carto-positron"
)

su.log_info(f"Created UserProfile: {custom_user.username}")
su.log_info(f"  Organization: {custom_user.organization}")
su.log_info(f"  Output format: {custom_user.default_output_format}")
su.log_info(f"  Map style: {custom_user.preferred_map_style}")

## 4. Client Branding Configuration

Each client can have custom branding with colors, fonts, and layout settings.

In [ ]:
# Create custom branding for a client
client_branding = BrandingConfig(
    primary_color="#1a4d2e",      # Forest green
    secondary_color="#4a7c59",    # Sage green
    accent_color="#f4a261",       # Warm orange
    text_color="#2d3436",         # Dark gray
    background_color="#ffffff",   # White
    primary_font="Helvetica",
    secondary_font="Georgia",
    header_height=50,
    footer_height=25
)

su.log_info("Created BrandingConfig")
su.log_info(f"  Primary color: {client_branding.primary_color}")
su.log_info(f"  Secondary color: {client_branding.secondary_color}")
su.log_info(f"  Accent color: {client_branding.accent_color}")
su.log_info(f"  Primary font: {client_branding.primary_font}")

# Get the color scheme as a dict
colors = client_branding.get_color_scheme()
su.log_debug(f"Color scheme dict: {colors}")

## 5. Validation Examples

Pydantic validates all fields with clear error messages.

In [ ]:
# Validation example: Email format
su.log_info("Testing email validation...")
try:
    valid_user = UserProfile(username="test", email="valid@example.com")
    su.log_info(f"  Valid email accepted: {valid_user.email}")
except ValueError as e:
    su.log_error(f"  Unexpected error: {e}")

try:
    invalid_user = UserProfile(username="test", email="not-an-email")
    su.log_warning("  Invalid email was not rejected!")
except ValueError as e:
    su.log_info(f"  Invalid email correctly rejected")

# Validation example: Hex color format
su.log_info("Testing hex color validation...")
try:
    valid_branding = BrandingConfig(
        primary_color="#FF5733",
        secondary_color="#33FF57",
        accent_color="#3357FF",
        text_color="#000000",
        background_color="#FFFFFF",
        primary_font="Arial",
        secondary_font="Times"
    )
    su.log_info(f"  Valid hex colors accepted: {valid_branding.primary_color}")
except ValueError as e:
    su.log_error(f"  Unexpected error: {e}")

try:
    invalid_branding = BrandingConfig(
        primary_color="red",  # Not hex format
        secondary_color="#33FF57",
        accent_color="#3357FF",
        text_color="#000000",
        background_color="#FFFFFF",
        primary_font="Arial",
        secondary_font="Times"
    )
    su.log_warning("  Invalid color was not rejected!")
except ValueError as e:
    su.log_info("  Invalid color 'red' correctly rejected (not hex format)")

## 6. File System Utilities

The `files/` module provides path management and file operations used throughout the library.

In [ ]:
# --- Path utilities (siege_utilities.files.paths) ---
from siege_utilities.files.paths import (
    ensure_path_exists,
    normalize_path,
    find_files_by_pattern,
    create_backup_path,
    get_file_extension,
    get_file_name_without_extension,
    is_hidden_file,
    get_relative_path,
)

# 6a. Ensure a directory exists (creates it if missing)
demo_dir = ensure_path_exists(OUTPUT_DIR / "demo_files")
su.log_info(f"Ensured directory: {demo_dir}")

# 6b. Normalize a path (resolve ~, .., symlinks)
raw_path = "~/Downloads/../Documents/./file.txt"
norm = normalize_path(raw_path)
su.log_info(f"normalize_path('{raw_path}') → {norm}")

# 6c. File extension helpers
for sample in ["report.pdf", "archive.tar.gz", "README"]:
    ext = get_file_extension(sample)
    stem = get_file_name_without_extension(sample)
    su.log_info(f"  '{sample}' → stem='{stem}', ext='{ext}'")

# 6d. Hidden-file detection
for sample in [".gitignore", "config.yaml", ".env"]:
    su.log_info(f"  is_hidden_file('{sample}') → {is_hidden_file(sample)}")

# 6e. Create a backup path
backup = create_backup_path(OUTPUT_DIR / "config.yaml", backup_suffix=".bak")
su.log_info(f"Backup path: {backup}")

# 6f. Relative path computation
rel = get_relative_path(OUTPUT_DIR, OUTPUT_DIR / "demo_files" / "subdir" / "file.txt")
su.log_info(f"Relative path: {rel}")

In [ ]:
# --- File operations (siege_utilities.files.operations) ---
from siege_utilities.files.operations import (
    file_exists,
    touch_file,
    count_lines,
    copy_file,
    get_file_size,
    list_directory,
)

# 6g. Touch a file (create if missing, update mtime if exists)
sample_file = OUTPUT_DIR / "demo_files" / "sample.txt"
touch_file(sample_file)
su.log_info(f"Touched file: {sample_file}")

# Write some content so we can demo count_lines and get_file_size
sample_file.write_text("line 1\nline 2\nline 3\nline 4\nline 5\n")

# 6h. Check file existence
su.log_info(f"file_exists('{sample_file.name}') → {file_exists(sample_file)}")
su.log_info(f"file_exists('nonexistent.txt') → {file_exists(OUTPUT_DIR / 'nonexistent.txt')}")

# 6i. Count lines in a file
lines = count_lines(sample_file)
su.log_info(f"count_lines('{sample_file.name}') → {lines}")

# 6j. Get file size in bytes
size = get_file_size(sample_file)
su.log_info(f"get_file_size('{sample_file.name}') → {size} bytes")

# 6k. Copy a file
copy_dest = OUTPUT_DIR / "demo_files" / "sample_copy.txt"
copy_file(sample_file, copy_dest, overwrite=True)
su.log_info(f"Copied → {copy_dest.name}")

# 6l. List directory contents
items = list_directory(OUTPUT_DIR / "demo_files")
su.log_info(f"list_directory('demo_files') → {[p.name for p in items]}")

# 6m. Find files by pattern
found = find_files_by_pattern(OUTPUT_DIR / "demo_files", "*.txt")
su.log_info(f"find_files_by_pattern('*.txt') → {[p.name for p in found]}")

## Summary

This notebook demonstrated:

| Feature | Description |
|---------|-------------|
| `HydraConfigManager` | Context manager for loading hierarchical configs |
| `UserProfile` | User preferences and API keys |
| `ClientProfile` | Client-specific settings |
| `BrandingConfig` | Colors, fonts, and layout |
| Validation | Automatic type and format validation |
| `files.paths` | Path normalization, extension helpers, backup paths, file search |
| `files.operations` | File existence, touch, copy, line count, size, directory listing |

**Logging Functions Used:**
- `su.configure_shared_logging(level)` - Initialize logging
- `su.log_info(msg)` - Informational messages
- `su.log_debug(msg)` - Debug details
- `su.log_warning(msg)` - Warnings
- `su.log_error(msg)` - Errors

**Next Steps:**
- See `02_Create_User_Client_Profiles.ipynb` for detailed profile creation
- See `03_Person_Actor_Architecture.ipynb` for the Person/Actor model
- See `10_Profile_Branding_Testing.ipynb` for 1Password integration

## Related
- Source: `siege_utilities/config/`
- Tests: `tests/test_config*.py`
- Docs: `docs/ARCHITECTURE.md`
